In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

import sys
sys.path.append('../src')

from preprocessing import (
    create_medication_df,
    create_notes_df,
    create_static_df,
    create_ts_data,
    create_vitals_df,
    get_dfs,
    get_valid_patient_ids,
)

In [ ]:
# Same root logic used in existing notebooks (run from notebooks/)
project_root = Path(os.path.dirname(os.getcwd()))
project_root

In [ ]:
dfs = get_dfs(str(project_root))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(
    vitals_ca=vitals_ca,
    vitals_lab=vitals_lab,
    medication=medication_df,
    merge_lab=True,
    merge_med=True,
    static_df=static_df,
)

notes = create_notes_df(
    dfs,
    filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy',
)

print('static_df shape:', static_df.shape)
print('ts_data shape:', ts_data.shape)
print('notes shape:', notes.shape)

In [ ]:
# Proposal constants
SEED = 42
MIN_TS_COUNT = 10
REQUIRE_NOTES = True
MIN_TIMELINE_DAYS = 545
POOL_B_SIZE = 550
POOL_C_SIZE = 50

In [ ]:
eligible_patient_ids = get_valid_patient_ids(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    min_ts_count=MIN_TS_COUNT,
    require_notes=REQUIRE_NOTES,
)
eligible_patient_ids = np.asarray(eligible_patient_ids, dtype=int)

max_timeline = ts_data.groupby('patient_id')['rel_days'].max()
eligible_timeline_days = max_timeline.reindex(eligible_patient_ids).fillna(-1).astype(int)

timeline_stats = eligible_timeline_days.describe(percentiles=[0.25, 0.5, 0.75])
print('Eligible patients:', len(eligible_patient_ids))
print('Timeline stats (days):')
print(timeline_stats[['min', '25%', '50%', '75%', 'max']])

print(f"Patients with >= 365 days: {(eligible_timeline_days >= 365).sum()}")
print(f"Patients with >= 545 days: {(eligible_timeline_days >= 545).sum()}")
print(f"Patients with >= 720 days: {(eligible_timeline_days >= 720).sum()}")
print(f"Patients with >= 1080 days: {(eligible_timeline_days >= 1080).sum()}")

In [ ]:
long_mask = eligible_timeline_days.values >= MIN_TIMELINE_DAYS
long_timeline = eligible_patient_ids[long_mask]
short_timeline = eligible_patient_ids[~long_mask]

required_long = POOL_C_SIZE + POOL_B_SIZE
if len(long_timeline) < required_long:
    raise ValueError(
        f"Not enough >= {MIN_TIMELINE_DAYS} day patients: need {required_long}, got {len(long_timeline)}"
    )

rng = np.random.default_rng(SEED)
long_shuffled = long_timeline.copy()
rng.shuffle(long_shuffled)

pool_c = long_shuffled[:POOL_C_SIZE]
pool_b = long_shuffled[POOL_C_SIZE:POOL_C_SIZE + POOL_B_SIZE]
pool_a_from_long = long_shuffled[POOL_C_SIZE + POOL_B_SIZE:]
pool_a = np.concatenate([pool_a_from_long, short_timeline])

print('Pool sizes:')
print('pool_a:', len(pool_a))
print('pool_b:', len(pool_b))
print('pool_c:', len(pool_c))
print('total:', len(pool_a) + len(pool_b) + len(pool_c))

In [ ]:
# Sanity checks: disjoint + full partition of eligible patients
set_a = set(pool_a.tolist())
set_b = set(pool_b.tolist())
set_c = set(pool_c.tolist())
set_all = set(eligible_patient_ids.tolist())

assert set_a.isdisjoint(set_b), 'pool_a overlaps pool_b'
assert set_a.isdisjoint(set_c), 'pool_a overlaps pool_c'
assert set_b.isdisjoint(set_c), 'pool_b overlaps pool_c'
assert (set_a | set_b | set_c) == set_all, 'Pools do not cover exactly all eligible IDs'

print('Disjointness and coverage checks passed.')

In [ ]:
output_dir = project_root / 'data' / 'splits'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'pool_assignments.json'

payload = {
    'pool_a': [int(pid) for pid in pool_a.tolist()],
    'pool_b': [int(pid) for pid in pool_b.tolist()],
    'pool_c': [int(pid) for pid in pool_c.tolist()],
    'seed': int(SEED),
    'min_timeline_days': int(MIN_TIMELINE_DAYS),
    'min_ts_count': int(MIN_TS_COUNT),
    'require_notes': bool(REQUIRE_NOTES),
}

with output_path.open('w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)

print(f'Saved: {output_path}')

In [ ]:
# Quick load-back validation
with (project_root / 'data' / 'splits' / 'pool_assignments.json').open('r', encoding='utf-8') as f:
    reloaded = json.load(f)

print({k: len(v) if isinstance(v, list) else v for k, v in reloaded.items()})